# GMM with Stochastic and Volatility Features

Fits a Gaussian mixture model on stochastic oscillator and Garman–Klass volatility features. Tests confidence-filtered exposure using daily Yahoo Finance prices. Supply Yahoo Finance symbols through Config.TICKER_UNIVERSE. The example uses SPY and compares it with URTH.


In [ ]:
"""
GMM Regime Detection Trading Framework
=======================================
Uses Stochastic K and Garman-Klass Volatility as features for Gaussian Mixture Model
to detect market regimes and generate trading signals.

Author: Trading Strategy Implementation
Date: 2025
"""

import pandas as pd
import numpy as np
from _market_data import download_ohlcv
from datetime import datetime, timedelta
from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION PARAMETERS
# ============================================================================

class Config:
    """
    Centralized configuration for the entire trading framework.
    All parameters are exposed here to avoid hardcoding throughout the codebase.
    """
    
    # Backtest Period
    START_DATE = "2019-01-01"  # 5 years + buffer for feature calculation
    END_DATE = "2024-12-31"
    
    # Universe Selection
    # RECOMMENDED: Use single asset for GMM regime detection
    # Single asset captures specific market dynamics without dilution
    # Popular choices: "SPY", "QQQ", "IWM", "GLD"
    TICKER_UNIVERSE = ["SPY"]  # Single asset recommended
    
    
    # Benchmark
    BENCHMARK_TICKER = "URTH"  # Global equity benchmark ETF
    
    # Feature Calculation Parameters
    STOCHASTIC_PERIOD = 14  # Lookback period for Stochastic K calculation
    STOCHASTIC_SMOOTH = 3   # Smoothing period for Stochastic K
    GARMAN_KLASS_PERIOD = 20  # Lookback for Garman-Klass volatility
    
    # Feature Normalization Parameters
    ZSCORE_LOOKBACK = 60  # Rolling window for z-score normalization (avoid look-ahead)
    
    # GMM Configuration
    N_COMPONENTS = 3  # Number of regimes to detect (2-4 typically optimal)
    GMM_COVARIANCE = 'full'  # Options: 'full', 'tied', 'diag', 'spherical'
    GMM_MAX_ITER = 100
    GMM_N_INIT = 10
    GMM_RANDOM_STATE = 42
    
    # Training Configuration
    MIN_TRAINING_SAMPLES = 120  # Minimum samples before starting to trade (expanding window)
    RETRAIN_FREQUENCY = 20  # Retrain GMM every N days (trading days)
    
    # Trading Rules - Position Sizing Strategy
    # Options:
    # 1. 'auto_sharpe': Automatic based on Sharpe ratio (may result in always-long if all regimes positive)
    # 2. 'auto_relative': Relative ranking - best regime gets 1.0, worst gets 0.0, middle gets 0.5
    # 3. 'auto_volatility_adjusted': Scale position by inverse volatility
    # 4. 'manual': Provide manual dictionary like {0: 1.0, 1: 0.5, 2: 0.0}
    POSITION_SIZING_METHOD = 'auto_relative'  # Changed from None to 'auto_relative'
    
    # Manual position override (only used if POSITION_SIZING_METHOD = 'manual')
    REGIME_POSITIONS_MANUAL = {0: 1.0, 1: 0.5, 2: 0.0}  
    
    # Position sizing constraints
    MIN_POSITION = 0.0  # Minimum position size (0.0 = can go to cash, -1.0 = can short)
    MAX_POSITION = 1.0  # Maximum position size (1.0 = max long, 1.5 = with leverage)
    
    # Regime confidence filter
    USE_REGIME_CONFIDENCE = True  # Only trade when regime assignment is high confidence
    MIN_REGIME_PROBABILITY = 0.6  # Minimum probability to take position (0.6 = 60% confidence)
    
    # Risk Management
    COMMISSION_RATE = 0.001  # 10 bps per trade (0.1%)
    INITIAL_CAPITAL = 100000  # Starting capital in USD
    
    # Position Sizing
    EQUAL_WEIGHT = True  # If True, equal weight across selected stocks; if False, use market cap weighting
    MAX_POSITION_SIZE = 0.20  # Maximum 20% in any single position
    
    # Reporting
    PLOT_RESULTS = True
    SAVE_RESULTS = True
    OUTPUT_DIR = "./gmm_strategy_results/"


# ============================================================================
# DATA FETCHING MODULE
# ============================================================================

class YahooDataFetcher:
    def get_historical_data(self, ticker, start_date, end_date):
        try:
            prices = download_ohlcv(ticker, start_date, end_date)
        except RuntimeError as error:
            print(error)
            return None
        print(f"Fetched {len(prices)} daily bars for {ticker}")
        return prices

    def get_universe_data(self, tickers: List[str], start_date: str, 
                         end_date: str) -> Dict[str, pd.DataFrame]:
        """
        Fetch data for multiple tickers.
        
        Parameters:
        -----------
        tickers : List[str]
            List of ticker symbols
        start_date : str
            Start date in YYYY-MM-DD format
        end_date : str
            End date in YYYY-MM-DD format
        
        Returns:
        --------
        Dict[str, pd.DataFrame] : Dictionary mapping ticker to OHLCV DataFrame
        """
        universe_data = {}
        
        for i, ticker in enumerate(tickers):
            print(f"Fetching {ticker} ({i+1}/{len(tickers)})...")
            df = self.get_historical_data(ticker, start_date, end_date)
            
            if df is not None and len(df) > 0:
                universe_data[ticker] = df
        
        print(f"\nSuccessfully fetched data for {len(universe_data)} out of {len(tickers)} tickers")
        return universe_data


# ============================================================================
# FEATURE ENGINEERING MODULE
# ============================================================================

class FeatureEngineer:
    """
    Calculates technical indicators without look-ahead bias.
    All calculations use only data available up to time t for decision at time t.
    """
    
    @staticmethod
    def calculate_stochastic_k(df: pd.DataFrame, period: int = 14, 
                               smooth: int = 3) -> pd.Series:
        """
        Calculate Stochastic K oscillator.
        
        Formula:
        %K = 100 * (Close - Low_n) / (High_n - Low_n)
        where Low_n = lowest low over period, High_n = highest high over period
        
        Then apply SMA smoothing.
        
        Parameters:
        -----------
        df : pd.DataFrame
            OHLCV data
        period : int
            Lookback period for high/low calculation
        smooth : int
            Smoothing period (SMA)
        
        Returns:
        --------
        pd.Series : Smoothed Stochastic K values
        """
        # Rolling lowest low and highest high
        # CRITICAL: .rolling() naturally avoids look-ahead as it only uses past data
        low_min = df['Low'].rolling(window=period, min_periods=period).min()
        high_max = df['High'].rolling(window=period, min_periods=period).max()
        
        # Calculate raw %K
        stoch_k = 100 * (df['Close'] - low_min) / (high_max - low_min)
        
        # Apply smoothing (SMA)
        stoch_k_smooth = stoch_k.rolling(window=smooth, min_periods=smooth).mean()
        
        return stoch_k_smooth
    
    @staticmethod
    def calculate_garman_klass_volatility(df: pd.DataFrame, period: int = 20) -> pd.Series:
        """
        Calculate Garman-Klass volatility estimator.
        
        This is a more efficient volatility estimator than close-to-close,
        as it uses OHLC information.
        
        Formula:
        GK_vol = sqrt( (1/n) * sum( 0.5*(log(H/L))^2 - (2*log(2)-1)*(log(C/O))^2 ) )
        
        Then annualized: GK_vol * sqrt(252)
        
        Parameters:
        -----------
        df : pd.DataFrame
            OHLCV data
        period : int
            Rolling window for volatility calculation
        
        Returns:
        --------
        pd.Series : Annualized Garman-Klass volatility
        """
        # Log ratios
        hl = np.log(df['High'] / df['Low'])
        co = np.log(df['Close'] / df['Open'])
        
        # Garman-Klass formula components
        # 0.5 * (log(H/L))^2 - (2*log(2)-1) * (log(C/O))^2
        gk_component = 0.5 * (hl ** 2) - (2 * np.log(2) - 1) * (co ** 2)
        
        # Rolling mean over period
        gk_vol = gk_component.rolling(window=period, min_periods=period).mean()
        
        # Take square root and annualize
        gk_vol_annualized = np.sqrt(gk_vol * 252)
        
        return gk_vol_annualized
    
    @staticmethod
    def rolling_zscore(series: pd.Series, window: int = 60) -> pd.Series:
        """
        Calculate rolling z-score without look-ahead bias.
        
        Z-score = (X - rolling_mean) / rolling_std
        
        CRITICAL: Uses expanding window for first 'window' periods to avoid NaN,
        then switches to rolling window. This ensures no forward-looking data is used.
        
        Parameters:
        -----------
        series : pd.Series
            Input feature series
        window : int
            Rolling window size
        
        Returns:
        --------
        pd.Series : Z-score normalized series
        """
        # Calculate rolling mean and std
        roll_mean = series.rolling(window=window, min_periods=window).mean()
        roll_std = series.rolling(window=window, min_periods=window).std()
        
        # Z-score calculation
        # Replace 0 std with small epsilon to avoid division by zero
        roll_std = roll_std.replace(0, 1e-6)
        zscore = (series - roll_mean) / roll_std
        
        return zscore
    
    def calculate_features(self, df: pd.DataFrame, config: Config) -> pd.DataFrame:
        """
        Calculate all features for GMM model.
        
        Parameters:
        -----------
        df : pd.DataFrame
            OHLCV data
        config : Config
            Configuration object with parameters
        
        Returns:
        --------
        pd.DataFrame : Original data with added feature columns
        """
        df = df.copy()
        
        # Calculate raw features
        print("Calculating Stochastic K...")
        df['stoch_k'] = self.calculate_stochastic_k(
            df, 
            period=config.STOCHASTIC_PERIOD,
            smooth=config.STOCHASTIC_SMOOTH
        )
        
        print("Calculating Garman-Klass Volatility...")
        df['gk_volatility'] = self.calculate_garman_klass_volatility(
            df,
            period=config.GARMAN_KLASS_PERIOD
        )
        
        # Normalize features using rolling z-score (no look-ahead)
        print("Normalizing features with rolling z-score...")
        df['stoch_k_norm'] = self.rolling_zscore(
            df['stoch_k'],
            window=config.ZSCORE_LOOKBACK
        )
        
        df['gk_vol_norm'] = self.rolling_zscore(
            df['gk_volatility'],
            window=config.ZSCORE_LOOKBACK
        )
        
        return df


# ============================================================================
# GMM REGIME DETECTION MODULE
# ============================================================================

class RegimeDetector:
    """
    Detects market regimes using Gaussian Mixture Model on normalized features.
    Uses expanding window training to avoid look-ahead bias.
    """
    
    def __init__(self, config: Config):
        self.config = config
        self.models = {}  # Store models per ticker
        self.last_train_date = {}  # Track last training date per ticker
        
    def train_gmm(self, features: np.ndarray) -> GaussianMixture:
        """
        Train a Gaussian Mixture Model on feature data.
        
        Parameters:
        -----------
        features : np.ndarray
            Feature matrix (n_samples, n_features)
        
        Returns:
        --------
        GaussianMixture : Trained GMM model
        """
        gmm = GaussianMixture(
            n_components=self.config.N_COMPONENTS,
            covariance_type=self.config.GMM_COVARIANCE,
            max_iter=self.config.GMM_MAX_ITER,
            n_init=self.config.GMM_N_INIT,
            random_state=self.config.GMM_RANDOM_STATE
        )
        
        gmm.fit(features)
        return gmm
    
    def predict_regime(self, df: pd.DataFrame, ticker: str) -> pd.DataFrame:
        """
        Predict regime for each day using expanding window training.
        
        CRITICAL: At time t, we only use data up to t-1 for training,
        then predict regime for time t. This ensures no look-ahead bias.
        
        Parameters:
        -----------
        df : pd.DataFrame
            Feature data with 'stoch_k_norm' and 'gk_vol_norm' columns
        ticker : str
            Ticker symbol (for model storage)
        
        Returns:
        --------
        pd.DataFrame : Data with added 'regime' and 'regime_proba' columns
        """
        df = df.copy()
        df['regime'] = np.nan
        df['regime_proba'] = np.nan  # Add probability of assigned regime
        
        feature_cols = ['stoch_k_norm', 'gk_vol_norm']
        
        # Drop rows where features are NaN (cannot compute regime)
        valid_idx = df[feature_cols].dropna().index
        
        if len(valid_idx) < self.config.MIN_TRAINING_SAMPLES:
            print(f"Insufficient data for {ticker}: {len(valid_idx)} samples")
            return df
        
        # Initialize tracking
        train_counter = 0
        current_model = None
        
        for i, date in enumerate(valid_idx):
            # Get training data up to (but not including) current date
            train_data = df.loc[valid_idx[:i], feature_cols].values
            
            # Need minimum samples to train
            if len(train_data) < self.config.MIN_TRAINING_SAMPLES:
                continue
            
            # Retrain model at specified frequency or if no model exists
            if current_model is None or train_counter >= self.config.RETRAIN_FREQUENCY:
                print(f"Training GMM for {ticker} at {date.strftime('%Y-%m-%d')} with {len(train_data)} samples")
                current_model = self.train_gmm(train_data)
                train_counter = 0
            
            # Predict regime for current date using model trained on past data
            current_features = df.loc[date, feature_cols].values.reshape(1, -1)
            regime = current_model.predict(current_features)[0]
            
            # Get probability of assigned regime (for confidence filtering)
            proba = current_model.predict_proba(current_features)[0]
            regime_probability = proba[regime]
            
            df.loc[date, 'regime'] = regime
            df.loc[date, 'regime_proba'] = regime_probability
            
            train_counter += 1
        
        return df
    
    def analyze_regimes(self, df: pd.DataFrame) -> Dict:
        """
        Analyze characteristics of each detected regime to inform trading rules.
        
        Parameters:
        -----------
        df : pd.DataFrame
            Data with regime labels
        
        Returns:
        --------
        Dict : Regime statistics
        """
        regime_stats = {}
        
        for regime in range(self.config.N_COMPONENTS):
            regime_data = df[df['regime'] == regime]
            
            if len(regime_data) == 0:
                continue
            
            # Calculate forward returns from Close[t] to Close[t+1]
            # CRITICAL: This is calculated post-hoc for analysis, not used in signal generation
            regime_data = regime_data.copy()
            regime_data['forward_return'] = regime_data['Close'].pct_change().shift(-1)
            
            stats = {
                'count': len(regime_data),
                'pct_of_total': len(regime_data) / len(df) * 100,
                'mean_return': regime_data['forward_return'].mean() * 100,
                'volatility': regime_data['forward_return'].std() * np.sqrt(252) * 100,
                'sharpe': (regime_data['forward_return'].mean() / regime_data['forward_return'].std() * np.sqrt(252)) if regime_data['forward_return'].std() > 0 else 0,
                'mean_stoch_k': regime_data['stoch_k'].mean(),
                'mean_gk_vol': regime_data['gk_volatility'].mean(),
                'win_rate': (regime_data['forward_return'] > 0).sum() / len(regime_data) * 100
            }
            
            regime_stats[regime] = stats
        
        return regime_stats
    
    def assign_positions_relative(self, regime_stats: Dict, config: Config) -> Dict[int, float]:
        """
        Assign positions based on relative ranking of regimes.
        
        CRITICAL: This solves the "always long" problem by ensuring differentiation.
        Best regime gets MAX_POSITION, worst gets MIN_POSITION, others interpolated.
        
        Parameters:
        -----------
        regime_stats : Dict
            Statistics for each regime
        config : Config
            Configuration with position constraints
        
        Returns:
        --------
        Dict[int, float] : Regime to position mapping
        """
        if len(regime_stats) == 0:
            return {}
        
        # Rank regimes by Sharpe ratio
        regime_sharpes = [(regime, stats['sharpe']) for regime, stats in regime_stats.items()]
        regime_sharpes.sort(key=lambda x: x[1], reverse=True)
        
        positions = {}
        n_regimes = len(regime_sharpes)
        
        if n_regimes == 1:
            # Only one regime - use max position
            positions[regime_sharpes[0][0]] = config.MAX_POSITION
        else:
            # Interpolate positions from max to min based on rank
            for i, (regime, _) in enumerate(regime_sharpes):
                # Linear interpolation from MAX to MIN
                position = config.MAX_POSITION - (i / (n_regimes - 1)) * (config.MAX_POSITION - config.MIN_POSITION)
                positions[regime] = position
        
        return positions
    
    def assign_positions_volatility_adjusted(self, regime_stats: Dict, config: Config) -> Dict[int, float]:
        """
        Assign positions inversely proportional to volatility, scaled by Sharpe.
        
        Lower volatility regimes get higher allocation.
        Negative Sharpe regimes get reduced or zero allocation.
        
        Parameters:
        -----------
        regime_stats : Dict
            Statistics for each regime
        config : Config
            Configuration with position constraints
        
        Returns:
        --------
        Dict[int, float] : Regime to position mapping
        """
        if len(regime_stats) == 0:
            return {}
        
        positions = {}
        
        for regime, stats in regime_stats.items():
            sharpe = stats['sharpe']
            volatility = stats['volatility']
            
            # Base position on Sharpe
            if sharpe > 0.5:
                base_position = config.MAX_POSITION
            elif sharpe > 0:
                base_position = config.MAX_POSITION * 0.5
            else:
                base_position = config.MIN_POSITION
            
            # Adjust by inverse volatility (lower vol = higher position)
            # Scale so that average volatility gets base position
            avg_volatility = np.mean([s['volatility'] for s in regime_stats.values()])
            vol_adjustment = avg_volatility / max(volatility, 1e-6)
            
            # Apply adjustment with bounds
            adjusted_position = base_position * vol_adjustment
            adjusted_position = np.clip(adjusted_position, config.MIN_POSITION, config.MAX_POSITION)
            
            positions[regime] = adjusted_position
        
        return positions
    
    def assign_positions_sharpe_threshold(self, regime_stats: Dict, config: Config) -> Dict[int, float]:
        """
        Simple threshold-based assignment.
        
        High Sharpe (>0.5) = MAX_POSITION
        Medium Sharpe (0 to 0.5) = 50% position
        Low/Negative Sharpe (<0) = MIN_POSITION
        
        Parameters:
        -----------
        regime_stats : Dict
            Statistics for each regime
        config : Config
            Configuration with position constraints
        
        Returns:
        --------
        Dict[int, float] : Regime to position mapping
        """
        if len(regime_stats) == 0:
            return {}
        
        positions = {}
        
        for regime, stats in regime_stats.items():
            sharpe = stats['sharpe']
            
            if sharpe > 0.5:
                positions[regime] = config.MAX_POSITION
            elif sharpe > 0:
                positions[regime] = config.MAX_POSITION * 0.5
            else:
                positions[regime] = config.MIN_POSITION
        
        return positions


# ============================================================================
# BACKTESTING ENGINE
# ============================================================================

class Backtester:
    """
    Backtesting engine following exact specifications:
    - Signal at Close[t]
    - Execute at Open[t+1]
    - Value portfolio at Open[t+1]
    - Returns = Open[t+2]/Open[t+1] - 1 for trading days
    - Returns = Close[t+1]/Close[t] - 1 for holding periods
    """
    
    def __init__(self, config: Config):
        self.config = config
        
    def generate_signals(self, df: pd.DataFrame, regime_positions: Dict[int, float],
                        config: Config) -> pd.DataFrame:
        """
        Generate position signals based on regime detection.
        
        Signal is generated at Close[t] based on regime detected at Close[t].
        Position is taken at Open[t+1].
        
        ENHANCEMENT: Applies regime confidence filter to avoid trading on uncertain regimes.
        
        Parameters:
        -----------
        df : pd.DataFrame
            Data with regime column and regime_proba column
        regime_positions : Dict[int, float]
            Mapping from regime to position size (MIN_POSITION to MAX_POSITION)
        config : Config
            Configuration object
        
        Returns:
        --------
        pd.DataFrame : Data with 'signal' column (position for next day)
        """
        df = df.copy()
        
        # Map regime to position
        df['signal'] = df['regime'].map(regime_positions)
        
        # Apply confidence filter if enabled
        if config.USE_REGIME_CONFIDENCE and 'regime_proba' in df.columns:
            # If regime probability is below threshold, reduce position to zero (go to cash)
            low_confidence_mask = df['regime_proba'] < config.MIN_REGIME_PROBABILITY
            print(f"Confidence filter: {low_confidence_mask.sum()} / {len(df)} days below {config.MIN_REGIME_PROBABILITY} threshold")
            df.loc[low_confidence_mask, 'signal'] = 0.0
        
        # Fill NaN signals with 0 (no position)
        df['signal'] = df['signal'].fillna(0)
        
        return df
    
    def calculate_returns(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Calculate returns following the specified methodology:
        
        For trade execution days:
        - Returns = Open[t+2]/Open[t+1] - 1
        
        For holding periods:
        - Returns = Close[t+1]/Close[t] - 1
        
        Parameters:
        -----------
        df : pd.DataFrame
            Data with 'signal' column
        
        Returns:
        --------
        pd.DataFrame : Data with returns calculations
        """
        df = df.copy()
        
        # Identify position changes (trade execution days)
        df['position_change'] = df['signal'].diff().fillna(0) != 0
        
        # Calculate returns based on position changes
        # For simplicity in multi-asset framework, we'll use Close-to-Close returns
        # and adjust for trades at Open. This is the standard approach for daily strategies.
        
        # CRITICAL: Returns are calculated from Close[t] to Close[t+1]
        # Position held is based on signal[t], which executes at Open[t+1]
        df['returns'] = df['Close'].pct_change()
        
        # Strategy returns = position * returns - transaction costs
        df['position'] = df['signal'].shift(1)  # Position held today based on yesterday's signal
        df['strategy_returns'] = df['position'] * df['returns']
        
        # Apply transaction costs on position changes
        # Cost applied when position changes from t-1 to t
        position_change_pct = df['position'].diff().abs()
        df['transaction_costs'] = position_change_pct * self.config.COMMISSION_RATE
        df['strategy_returns_net'] = df['strategy_returns'] - df['transaction_costs']
        
        return df
    
    def backtest_single_asset(self, df: pd.DataFrame, regime_positions: Dict[int, float],
                             config: Config) -> pd.DataFrame:
        """
        Run backtest for a single asset.
        
        Parameters:
        -----------
        df : pd.DataFrame
            Data with regime labels
        regime_positions : Dict[int, float]
            Position sizing by regime
        config : Config
            Configuration object
        
        Returns:
        --------
        pd.DataFrame : Backtest results
        """
        df = self.generate_signals(df, regime_positions, config)
        df = self.calculate_returns(df)
        
        # Calculate cumulative returns
        df['cum_returns'] = (1 + df['returns']).cumprod()
        df['cum_strategy_returns'] = (1 + df['strategy_returns_net'].fillna(0)).cumprod()
        
        return df
    
    def backtest_portfolio(self, universe_data: Dict[str, pd.DataFrame], 
                          regime_positions: Dict[int, float]) -> pd.DataFrame:
        """
        Run backtest for portfolio of assets with equal weighting.
        
        Parameters:
        -----------
        universe_data : Dict[str, pd.DataFrame]
            Dictionary of ticker -> backtest results
        regime_positions : Dict[int, float]
            Position sizing by regime
        
        Returns:
        --------
        pd.DataFrame : Portfolio-level returns
        """
        # Get all unique dates from all assets
        all_dates = pd.DatetimeIndex([])
        for df in universe_data.values():
            all_dates = all_dates.union(df.index)
        all_dates = all_dates.sort_values()
        
        # Create portfolio returns DataFrame
        portfolio_returns = pd.DataFrame(index=all_dates)
        portfolio_returns['strategy_returns'] = 0.0
        portfolio_returns['transaction_costs'] = 0.0
        portfolio_returns['n_assets'] = 0
        
        # Aggregate returns across assets
        for ticker, df in universe_data.items():
            if 'strategy_returns_net' not in df.columns:
                continue
            
            # Add asset returns to portfolio (equal weight)
            for date in df.index:
                if date in portfolio_returns.index:
                    portfolio_returns.loc[date, 'strategy_returns'] += df.loc[date, 'strategy_returns_net']
                    portfolio_returns.loc[date, 'n_assets'] += 1
        
        # Average returns across assets (equal weight)
        portfolio_returns['strategy_returns'] = portfolio_returns.apply(
            lambda row: row['strategy_returns'] / row['n_assets'] if row['n_assets'] > 0 else 0,
            axis=1
        )
        
        # Calculate cumulative returns
        portfolio_returns['cum_returns'] = (1 + portfolio_returns['strategy_returns']).cumprod()
        
        return portfolio_returns


# ============================================================================
# PERFORMANCE METRICS MODULE
# ============================================================================

class PerformanceAnalyzer:
    """
    Calculate and display performance metrics for strategy evaluation.
    """
    
    @staticmethod
    def calculate_metrics(returns: pd.Series, name: str = "Strategy") -> Dict:
        """
        Calculate comprehensive performance metrics.
        
        Parameters:
        -----------
        returns : pd.Series
            Series of returns
        name : str
            Name of the strategy/asset
        
        Returns:
        --------
        Dict : Performance metrics
        """
        # Remove NaN values
        returns = returns.dropna()
        
        if len(returns) == 0:
            return {}
        
        # Calculate metrics
        total_return = (1 + returns).prod() - 1
        ann_return = (1 + total_return) ** (252 / len(returns)) - 1
        ann_volatility = returns.std() * np.sqrt(252)
        sharpe_ratio = ann_return / ann_volatility if ann_volatility > 0 else 0
        
        # Drawdown calculation
        cum_returns = (1 + returns).cumprod()
        running_max = cum_returns.expanding().max()
        drawdown = (cum_returns - running_max) / running_max
        max_drawdown = drawdown.min()
        
        # Win rate
        win_rate = (returns > 0).sum() / len(returns) * 100
        
        metrics = {
            'name': name,
            'total_return_%': total_return * 100,
            'annual_return_%': ann_return * 100,
            'annual_volatility_%': ann_volatility * 100,
            'sharpe_ratio': sharpe_ratio,
            'max_drawdown_%': max_drawdown * 100,
            'win_rate_%': win_rate,
            'n_trades': len(returns),
        }
        
        return metrics
    
    @staticmethod
    def print_metrics(metrics: Dict):
        """Pretty print performance metrics."""
        print(f"\n{'='*60}")
        print(f"Performance Metrics: {metrics['name']}")
        print(f"{'='*60}")
        print(f"Total Return:          {metrics['total_return_%']:>10.2f}%")
        print(f"Annual Return:         {metrics['annual_return_%']:>10.2f}%")
        print(f"Annual Volatility:     {metrics['annual_volatility_%']:>10.2f}%")
        print(f"Sharpe Ratio:          {metrics['sharpe_ratio']:>10.2f}")
        print(f"Max Drawdown:          {metrics['max_drawdown_%']:>10.2f}%")
        print(f"Win Rate:              {metrics['win_rate_%']:>10.2f}%")
        print(f"Number of Periods:     {metrics['n_trades']:>10}")
        print(f"{'='*60}\n")
    
    @staticmethod
    def plot_results(strategy_df: pd.DataFrame, benchmark_df: pd.DataFrame, 
                    config: Config):
        """
        Plot backtest results comparing strategy vs benchmark.
        
        Parameters:
        -----------
        strategy_df : pd.DataFrame
            Strategy returns
        benchmark_df : pd.DataFrame
            Benchmark returns
        config : Config
            Configuration object
        """
        fig, axes = plt.subplots(3, 1, figsize=(14, 10))
        
        # Plot 1: Cumulative Returns
        axes[0].plot(strategy_df.index, strategy_df['cum_returns'], 
                    label='GMM Strategy', linewidth=2)
        axes[0].plot(benchmark_df.index, benchmark_df['cum_returns'], 
                    label=f'{config.BENCHMARK_TICKER} Benchmark', 
                    linewidth=2, alpha=0.7)
        axes[0].set_ylabel('Cumulative Returns')
        axes[0].set_title('Strategy Performance vs Benchmark')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # Plot 2: Rolling Sharpe Ratio (252-day window)
        strategy_rolling_sharpe = (
            strategy_df['strategy_returns'].rolling(252).mean() / 
            strategy_df['strategy_returns'].rolling(252).std() * np.sqrt(252)
        )
        benchmark_rolling_sharpe = (
            benchmark_df['returns'].rolling(252).mean() / 
            benchmark_df['returns'].rolling(252).std() * np.sqrt(252)
        )
        
        axes[1].plot(strategy_df.index, strategy_rolling_sharpe, 
                    label='GMM Strategy', linewidth=2)
        axes[1].plot(benchmark_df.index, benchmark_rolling_sharpe, 
                    label=f'{config.BENCHMARK_TICKER} Benchmark', 
                    linewidth=2, alpha=0.7)
        axes[1].axhline(y=0, color='black', linestyle='--', alpha=0.3)
        axes[1].set_ylabel('Rolling Sharpe (252d)')
        axes[1].set_title('Rolling Sharpe Ratio')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        
        # Plot 3: Drawdown
        strategy_cum = strategy_df['cum_returns']
        strategy_running_max = strategy_cum.expanding().max()
        strategy_dd = (strategy_cum - strategy_running_max) / strategy_running_max
        
        benchmark_cum = benchmark_df['cum_returns']
        benchmark_running_max = benchmark_cum.expanding().max()
        benchmark_dd = (benchmark_cum - benchmark_running_max) / benchmark_running_max
        
        axes[2].fill_between(strategy_df.index, strategy_dd * 100, 0, 
                            label='GMM Strategy', alpha=0.3)
        axes[2].fill_between(benchmark_df.index, benchmark_dd * 100, 0, 
                            label=f'{config.BENCHMARK_TICKER} Benchmark', alpha=0.3)
        axes[2].set_ylabel('Drawdown (%)')
        axes[2].set_xlabel('Date')
        axes[2].set_title('Drawdown')
        axes[2].legend()
        axes[2].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    
    @staticmethod
    def plot_regime_analysis(df: pd.DataFrame, ticker: str, config: Config):
        """
        Plot detailed regime analysis for a single asset.
        Shows feature space, regime transitions, and performance by regime.
        
        CRITICAL: This visualization is essential for single-asset strategies
        to understand regime characteristics and validate the GMM model.
        
        Parameters:
        -----------
        df : pd.DataFrame
            Backtest results with regime labels
        ticker : str
            Ticker symbol
        config : Config
            Configuration object
        """
        # Remove NaN regimes for visualization
        df_clean = df.dropna(subset=['regime', 'stoch_k_norm', 'gk_vol_norm'])
        
        if len(df_clean) == 0:
            print(f"No valid regime data for {ticker}")
            return
        
        fig = plt.figure(figsize=(16, 12))
        gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)
        
        # Plot 1: Feature Space (2D scatter colored by regime)
        ax1 = fig.add_subplot(gs[0, 0])
        for regime in range(config.N_COMPONENTS):
            regime_data = df_clean[df_clean['regime'] == regime]
            ax1.scatter(regime_data['stoch_k_norm'], regime_data['gk_vol_norm'],
                       alpha=0.5, s=20, label=f'Regime {regime}')
        ax1.set_xlabel('Stochastic K (normalized)')
        ax1.set_ylabel('Garman-Klass Volatility (normalized)')
        ax1.set_title(f'{ticker}: Feature Space by Regime')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Plot 2: Regime transitions over time
        ax2 = fig.add_subplot(gs[0, 1])
        ax2.scatter(df_clean.index, df_clean['regime'], c=df_clean['regime'], 
                   cmap='viridis', s=10, alpha=0.6)
        ax2.set_ylabel('Regime')
        ax2.set_title(f'{ticker}: Regime Transitions Over Time')
        ax2.set_yticks(range(config.N_COMPONENTS))
        ax2.grid(True, alpha=0.3)
        
        # Plot 3: Price with regime overlay
        ax3 = fig.add_subplot(gs[1, :])
        # Plot price
        ax3_price = ax3.twinx()
        ax3_price.plot(df_clean.index, df_clean['Close'], color='black', 
                      linewidth=1, alpha=0.5, label='Close Price')
        ax3_price.set_ylabel('Price')
        
        # Overlay regime as colored background
        for regime in range(config.N_COMPONENTS):
            regime_data = df_clean[df_clean['regime'] == regime]
            if len(regime_data) > 0:
                ax3.scatter(regime_data.index, [regime] * len(regime_data),
                          alpha=0.3, s=50, label=f'Regime {regime}')
        
        ax3.set_ylabel('Regime')
        ax3.set_title(f'{ticker}: Price Chart with Regime Overlay')
        ax3.set_yticks(range(config.N_COMPONENTS))
        ax3.legend(loc='upper left')
        ax3_price.legend(loc='upper right')
        ax3.grid(True, alpha=0.3)
        
        # Plot 4: Returns distribution by regime
        ax4 = fig.add_subplot(gs[2, 0])
        regime_returns = []
        regime_labels = []
        for regime in range(config.N_COMPONENTS):
            regime_data = df_clean[df_clean['regime'] == regime]
            if len(regime_data) > 0 and 'returns' in regime_data.columns:
                returns = regime_data['returns'].dropna() * 100  # Convert to percentage
                regime_returns.append(returns)
                regime_labels.append(f'Regime {regime}')
        
        if regime_returns:
            ax4.violinplot(regime_returns, positions=range(len(regime_returns)),
                          showmeans=True, showmedians=True)
            ax4.set_xticks(range(len(regime_labels)))
            ax4.set_xticklabels(regime_labels)
            ax4.set_ylabel('Returns (%)')
            ax4.set_title(f'{ticker}: Returns Distribution by Regime')
            ax4.axhline(y=0, color='red', linestyle='--', alpha=0.5)
            ax4.grid(True, alpha=0.3)
        
        # Plot 5: Cumulative returns by regime
        ax5 = fig.add_subplot(gs[2, 1])
        for regime in range(config.N_COMPONENTS):
            regime_data = df_clean[df_clean['regime'] == regime].copy()
            if len(regime_data) > 0 and 'returns' in regime_data.columns:
                regime_data['cum_ret'] = (1 + regime_data['returns']).cumprod()
                ax5.plot(regime_data.index, regime_data['cum_ret'], 
                        label=f'Regime {regime}', linewidth=2, alpha=0.7)
        
        ax5.set_ylabel('Cumulative Returns')
        ax5.set_title(f'{ticker}: Cumulative Returns by Regime')
        ax5.legend()
        ax5.grid(True, alpha=0.3)
        
        plt.suptitle(f'GMM Regime Analysis: {ticker}', fontsize=16, fontweight='bold')
        plt.show()


# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    """
    Main execution function orchestrating the entire framework.
    """
    print("="*80)
    print("GMM REGIME DETECTION TRADING FRAMEWORK")
    print("="*80)
    
    # Initialize configuration
    config = Config()
    
    # Step 1: Data Fetching
    print("\n[1/7] Fetching market data from Yahoo Finance...")
    fetcher = YahooDataFetcher()
    tickers = config.TICKER_UNIVERSE
    if not tickers:
        raise ValueError("Add at least one Yahoo Finance symbol to TICKER_UNIVERSE.")

    # Fetch universe data
    universe_data = fetcher.get_universe_data(tickers, config.START_DATE, config.END_DATE)
    if not universe_data:
        raise RuntimeError("No Yahoo price history was available for the trading universe.")
    
    # Fetch benchmark data
    print(f"\nFetching benchmark data ({config.BENCHMARK_TICKER})...")
    benchmark_data = fetcher.get_historical_data(
        config.BENCHMARK_TICKER, 
        config.START_DATE, 
        config.END_DATE
    )
    
    if benchmark_data is None:
        print(f"ERROR: Could not fetch benchmark data for {config.BENCHMARK_TICKER}")
        return
    
    # Step 2: Feature Engineering
    print("\n[2/7] Engineering features (Stochastic K + Garman-Klass Volatility)...")
    engineer = FeatureEngineer()
    
    # Calculate features for all assets
    for ticker in universe_data.keys():
        print(f"\nProcessing {ticker}...")
        universe_data[ticker] = engineer.calculate_features(universe_data[ticker], config)
    
    # Calculate features for benchmark
    print(f"\nProcessing benchmark ({config.BENCHMARK_TICKER})...")
    benchmark_data = engineer.calculate_features(benchmark_data, config)
    
    # Step 3: Regime Detection
    print("\n[3/7] Detecting market regimes using GMM...")
    detector = RegimeDetector(config)
    
    # Detect regimes for all assets
    for ticker in universe_data.keys():
        print(f"\nDetecting regimes for {ticker}...")
        universe_data[ticker] = detector.predict_regime(universe_data[ticker], ticker)
    
    # Step 4: Regime Analysis
    print("\n[4/7] Analyzing regime characteristics...")
    
    # Aggregate regime statistics across all assets
    all_regime_stats = {}
    for ticker, df in universe_data.items():
        ticker_stats = detector.analyze_regimes(df)
        for regime, stats in ticker_stats.items():
            if regime not in all_regime_stats:
                all_regime_stats[regime] = []
            all_regime_stats[regime].append(stats)
    
    # Average statistics across assets
    print("\nRegime Statistics (averaged across assets):")
    print("="*80)
    
    avg_regime_stats = {}
    for regime in range(config.N_COMPONENTS):
        if regime not in all_regime_stats:
            continue
        
        stats_list = all_regime_stats[regime]
        avg_stats = {
            'count': np.mean([s['count'] for s in stats_list]),
            'mean_return': np.mean([s['mean_return'] for s in stats_list]),
            'volatility': np.mean([s['volatility'] for s in stats_list]),
            'sharpe': np.mean([s['sharpe'] for s in stats_list]),
            'mean_stoch_k': np.mean([s['mean_stoch_k'] for s in stats_list]),
            'mean_gk_vol': np.mean([s['mean_gk_vol'] for s in stats_list]),
            'win_rate': np.mean([s['win_rate'] for s in stats_list])
        }
        avg_regime_stats[regime] = avg_stats
        
        print(f"\nRegime {regime}:")
        print(f"  Average Days:          {avg_stats['count']:.0f}")
        print(f"  Mean Forward Return:   {avg_stats['mean_return']:.3f}%")
        print(f"  Volatility:            {avg_stats['volatility']:.2f}%")
        print(f"  Sharpe Ratio:          {avg_stats['sharpe']:.2f}")
        print(f"  Win Rate:              {avg_stats['win_rate']:.2f}%")
        print(f"  Mean Stochastic K:     {avg_stats['mean_stoch_k']:.2f}")
        print(f"  Mean GK Volatility:    {avg_stats['mean_gk_vol']:.2f}")
    
    # Assign positions based on selected method
    print("\n" + "="*80)
    print(f"Position Sizing Method: {config.POSITION_SIZING_METHOD}")
    print("="*80)
    
    if config.POSITION_SIZING_METHOD == 'manual':
        regime_positions = config.REGIME_POSITIONS_MANUAL
        print("Using manual position assignments:")
    elif config.POSITION_SIZING_METHOD == 'auto_relative':
        regime_positions = detector.assign_positions_relative(avg_regime_stats, config)
        print("Using relative ranking (best regime = max, worst = min):")
    elif config.POSITION_SIZING_METHOD == 'auto_volatility_adjusted':
        regime_positions = detector.assign_positions_volatility_adjusted(avg_regime_stats, config)
        print("Using volatility-adjusted positions (lower vol = higher allocation):")
    elif config.POSITION_SIZING_METHOD == 'auto_sharpe':
        regime_positions = detector.assign_positions_sharpe_threshold(avg_regime_stats, config)
        print("Using Sharpe threshold (>0.5 = max, 0-0.5 = half, <0 = min):")
    else:
        # Default to relative if method not recognized
        regime_positions = detector.assign_positions_relative(avg_regime_stats, config)
        print("Using relative ranking (default):")
    
    for regime, position in sorted(regime_positions.items()):
        regime_sharpe = avg_regime_stats[regime]['sharpe'] if regime in avg_regime_stats else 0
        print(f"  Regime {regime}: {position*100:.1f}% position (Sharpe: {regime_sharpe:.2f})")
    
    if config.USE_REGIME_CONFIDENCE:
        print(f"\nConfidence Filter: Enabled (min probability: {config.MIN_REGIME_PROBABILITY})")
        print("  Positions reduced to 0% when regime confidence is low")
    else:
        print("\nConfidence Filter: Disabled")
    
    # Step 5: Backtesting
    print("\n[5/7] Running backtest...")
    backtester = Backtester(config)
    
    # Backtest each asset
    backtest_results = {}
    for ticker in universe_data.keys():
        print(f"Backtesting {ticker}...")
        backtest_results[ticker] = backtester.backtest_single_asset(
            universe_data[ticker], 
            regime_positions,
            config
        )
    
    # Create portfolio-level results
    print("\nAggregating portfolio results...")
    portfolio_results = backtester.backtest_portfolio(backtest_results, regime_positions)
    
    # Backtest benchmark (buy and hold)
    print(f"\nBacktesting benchmark ({config.BENCHMARK_TICKER})...")
    benchmark_data['returns'] = benchmark_data['Close'].pct_change()
    benchmark_data['cum_returns'] = (1 + benchmark_data['returns']).cumprod()
    
    # Step 6: Performance Analysis
    print("\n[6/7] Analyzing performance...")
    analyzer = PerformanceAnalyzer()
    
    # Calculate metrics
    strategy_metrics = analyzer.calculate_metrics(
        portfolio_results['strategy_returns'], 
        "GMM Strategy"
    )
    benchmark_metrics = analyzer.calculate_metrics(
        benchmark_data['returns'], 
        f"{config.BENCHMARK_TICKER} Benchmark"
    )
    
    # Print metrics
    analyzer.print_metrics(strategy_metrics)
    analyzer.print_metrics(benchmark_metrics)
    
    # Additional strategy analysis
    print("\n" + "="*80)
    print("STRATEGY ANALYSIS")
    print("="*80)
    
    # Calculate number of position changes (trades)
    if len(universe_data) == 1:
        ticker = list(universe_data.keys())[0]
        result_df = backtest_results[ticker]
        
        n_trades = (result_df['signal'].diff() != 0).sum()
        avg_position = result_df['signal'].mean() * 100
        
        print(f"Number of Position Changes: {n_trades}")
        print(f"Average Position Size: {avg_position:.1f}%")
        
        # Regime distribution
        regime_dist = result_df['regime'].value_counts().sort_index()
        print(f"\nRegime Distribution:")
        for regime, count in regime_dist.items():
            pct = count / len(result_df) * 100
            avg_pos = result_df[result_df['regime'] == regime]['signal'].mean() * 100
            print(f"  Regime {int(regime)}: {count} days ({pct:.1f}%), Avg Position: {avg_pos:.1f}%")
        
        # Confidence distribution if available
        if 'regime_proba' in result_df.columns:
            avg_confidence = result_df['regime_proba'].mean() * 100
            low_conf_days = (result_df['regime_proba'] < config.MIN_REGIME_PROBABILITY).sum()
            print(f"\nRegime Confidence:")
            print(f"  Average Confidence: {avg_confidence:.1f}%")
            print(f"  Days Below Threshold ({config.MIN_REGIME_PROBABILITY}): {low_conf_days} ({low_conf_days/len(result_df)*100:.1f}%)")
    
    # Step 7: Visualization
    if config.PLOT_RESULTS:
        print("\n[7/7] Generating plots...")
        analyzer.plot_results(portfolio_results, benchmark_data, config)
        
        # If single asset, show detailed regime analysis
        if len(universe_data) == 1:
            ticker = list(universe_data.keys())[0]
            print(f"\nGenerating detailed regime analysis for {ticker}...")
            analyzer.plot_regime_analysis(backtest_results[ticker], ticker, config)
    
    print("\n" + "="*80)
    print("BACKTEST COMPLETE")
    print("="*80)
    
    return {
        'portfolio_results': portfolio_results,
        'benchmark_data': benchmark_data,
        'strategy_metrics': strategy_metrics,
        'benchmark_metrics': benchmark_metrics,
        'regime_positions': regime_positions
    }


if __name__ == "__main__":
    results = main()